In [2]:
import psutil
import time
import numpy as np
import pandas as pd
import csv
import os
import optuna
from PIL import Image
from matplotlib import pylab as plt
import tifffile
import warnings
import geopandas as gpd
import random
import glob
from shapely.geometry import box
from rasterio.features import rasterize
from affine import Affine
from optuna.pruners import MedianPruner
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchmetrics.classification import BinaryAUROC
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from CNNtoolbox import *

In [ ]:
# ===== 再現性確保のためのシード固定 =====

SEED = 42

def set_seed(seed: int = SEED):
    """random / numpy / torch (CPU・GPU) のシードを固定し、cuDNN を決定論的にする。"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [4]:
dataset = pd.read_csv("../../../../くま発見3次メッシュデータセット_東北全県.csv")
dataset["kuma_view"] = 1*(dataset["kuma_view"]>0)

# ===== 実験2(持ち回り5fold): 共有5分割の割当(fold_group)を付与。割合は実験3の県別件数に一致 =====
dataset["fold_group"] = pd.read_csv("fold_assignment.csv")["fold_group"].values
assert dataset["fold_group"].notna().all(), "fold_group 欠損あり"


In [5]:
import torch
import torch.nn as nn


# ============================================================
# 1) Gradient Reversal Layer (GRL)
# ============================================================
class _GradReverseFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return _GradReverseFn.apply(x, lambda_)


# ============================================================
# 2) (元の) ResBlock
# ============================================================
class ResBlock(nn.Module):
    """stride 付き畳み込みで空間ダウンサンプリングする BasicBlock"""
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=stride, padding=2, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.pool  = nn.MaxPool2d(4)

        self.downsample = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.MaxPool2d(4)
        )
        #self.downsample = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0, bias=False)

    def forward(self, x):
        identity = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        out = self.pool(x)
        out = self.relu(out + self.downsample(identity))
        return out


# ============================================================
# 3) DANN版 CNN（構造を極力維持）
# ============================================================
class DANN_CNN(nn.Module):
    """
    - 入力: (x, y, z, B, C)
      * z はドメイン one-hot (教師) として使用し、タスク特徴には結合しない（=DANNの基本形）
    - 出力: (task_logits or prob, domain_logits)
    """
    def __init__(
        self,
        domain_classes: int,
        base_channels: int = 16,
        dropout: float = 0.0,
        m1: int = 1024,
        m2: int = 1024,
        apply_sigmoid: bool = False,   # 推奨: False（BCEWithLogitsLossを使う）
        domain_hidden: int = 256
    ):
        super().__init__()

        if domain_classes < 2:
            raise ValueError("domain_classes must be >= 2 for DANN.")
        self.domain_head = nn.Sequential(
            nn.Linear(m2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, domain_classes)   # logits (N, domain_classes)
        )
        # --- x,B,C ブランチ（元の形を維持）---
        # ※ 元コード同様 layer0 を ResBlock で上書きする形を踏襲
        self.layer0 = nn.Sequential(
            nn.Conv2d(3, base_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
        )
        self.layer0 = ResBlock(24, base_channels, stride=1)  # 元コード踏襲
        self.layer2 = ResBlock(base_channels, base_channels, stride=1)

        # --- y ブランチ（元の形を維持）---
        self.layerA = nn.Sequential(
            nn.Conv2d(in_channels=5, out_channels=5, kernel_size=4),
            nn.BatchNorm2d(5),
            nn.ReLU(),
            nn.MaxPool2d(4)
        )

        # --- 共有表現（元の fc, fcA を shared として扱う）---
        self.fc  = nn.Sequential(nn.LazyLinear(m1), nn.ReLU())
        self.fcA = nn.Sequential(nn.Linear(m1, m2), nn.ReLU())
        self.dropout = nn.Dropout(dropout)

        # --- タスクヘッド（元の fc2）---
        self.task_fc2 = nn.Linear(m2, 1)
        self.apply_sigmoid = apply_sigmoid
        self.sigmoid = nn.Sigmoid()

        # --- ドメインヘッド（新規）---
        # 共有表現 (m2) を入力に domain_classes を分類
        self.domain_head = nn.Sequential(
            nn.Linear(m2, domain_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(domain_hidden, domain_classes)
        )

    @staticmethod
    def z_to_domain_target(z: torch.Tensor) -> torch.Tensor:
        """
        z: one-hot (N, D) または (N, D, 1, 1) を想定
        戻り: (N,) の long（CrossEntropyLoss 用）
        """
        z = z.view(z.size(0), -1)
        # one-hot を想定：最大値indexがドメインID
        return torch.argmax(z, dim=1).long()

    def forward(self, x, y, z1, z, B, C,grl_lambda: float = 1.0, return_domain: bool = False, return_features: bool = False, **kwargs):                       # ★追加（DataParallel互換性）):
        # --- 画像側（元の処理を維持）---
        x = torch.cat([x, B, C], dim=1)
        x = self.layer0(x)
        x = self.layer2(x)

        y = self.layerA(y)

        x = x.view(x.size(0), -1)
        y = y.view(y.size(0), -1)
        z1 = z1.view(z1.size(0), -1)

        # --- DANN化の要点：z はタスク特徴に入れない ---
        feat = torch.cat([x, y, z1], dim=1)

        # --- 共有表現 ---
        h = self.fc(feat)
        h = self.dropout(h)
        h = self.fcA(h)
        h = self.dropout(h)  # shared feature (m2)

        # --- タスク予測 ---
        task_logits = self.task_fc2(h)
        if self.apply_sigmoid:
            task_out = self.sigmoid(task_logits)
        else:
            task_out = task_logits  # BCEWithLogitsLoss 前提

        # --- ドメイン予測（GRL付き）---
        rev_h = grad_reverse(h, lambda_=grl_lambda)
        domain_logits = self.domain_head(rev_h)  # CrossEntropyLoss 前提

        if return_features:
            return task_out, domain_logits, h
        if return_domain:
            return task_out, domain_logits
        return task_out

In [6]:
from itertools import cycle

def _domain_target_from_onehot(z_onehot: torch.Tensor) -> torch.Tensor:
    z = z_onehot.view(z_onehot.size(0), -1)
    return torch.argmax(z, dim=1).long()

def train(
    model, train_loader, criterion, device, optimizer,
    target_loader=None, domain_criterion=None, alpha: float = 0.5, grl_lambda: float = 1.0,
    use_amp: bool = True  # AMP（bfloat16混合精度）の有効/無効（H100はGradScaler不要）
):
    model.train()
    running_loss = 0
    correct = 0
    total = 0; positive = 0; sumout = 0; sumlabel = 0

    # DANN用：target iterator（長さ違いに備えてcycle）
    tgt_iter = cycle(target_loader) if target_loader is not None else None

    # AMP精度設定（H100はbfloat16推奨・GradScaler不要）
    _amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for images, images2, images3, images3B, images4B, images4C, labels in train_loader:
        images   = images.float().to(device, non_blocking=True)
        images2  = images2.float().to(device, non_blocking=True)
        images3  = images3.float().to(device, non_blocking=True)
        images3B = images3B.float().to(device, non_blocking=True)  # z (one-hot)  [BUG FIX: images3 -> images3B]
        images4B = images4B.float().to(device, non_blocking=True)
        images4C = images4C.float().to(device, non_blocking=True)
        labels   = labels.float().to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # [OPTIMIZED]

        # -------------------------
        # 従来モード（target無し）
        # -------------------------
        if target_loader is None:
            with torch.autocast(device_type='cuda', dtype=_amp_dtype, enabled=use_amp):
                outputs = model(images, images2, images3, images3B, images4B, images4C)
                loss = criterion(outputs.squeeze(1).float(), labels.float())

        # -------------------------
        # DANNモード（targetあり）
        # -------------------------
        else:
            if domain_criterion is None:
                raise ValueError("DANN mode requires domain_criterion (e.g., CrossEntropyLoss).")

            # target batch を先にGPUへ転送（autocastの外でOK）
            t_images, t_images2, t_images3, t_images3B, t_images4B, t_images4C, _t_labels = next(tgt_iter)
            t_images   = t_images.float().to(device, non_blocking=True)
            t_images2  = t_images2.float().to(device, non_blocking=True)
            t_images3  = t_images3.float().to(device, non_blocking=True)
            t_images3B = t_images3B.float().to(device, non_blocking=True)
            t_images4B = t_images4B.float().to(device, non_blocking=True)
            t_images4C = t_images4C.float().to(device, non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=_amp_dtype, enabled=use_amp):
                # source forward（task + domain）
                outputs, dom_logits_s = model(
                    images, images2, images3, images3B, images4B, images4C,
                    grl_lambda=grl_lambda, return_domain=True
                )
                task_loss = criterion(outputs.squeeze(1).float(), labels.float())

                dom_target_s = _domain_target_from_onehot(images3B)
                dom_loss_s   = domain_criterion(dom_logits_s, dom_target_s)

                _t_out, dom_logits_t = model(
                    t_images, t_images2, t_images3, t_images3B, t_images4B, t_images4C,
                    grl_lambda=grl_lambda, return_domain=True
                )
                dom_target_t = _domain_target_from_onehot(t_images3B)
                dom_loss_t   = domain_criterion(dom_logits_t, dom_target_t)

                loss = task_loss + alpha * (dom_loss_s + dom_loss_t)

        running_loss += loss.detach()
        loss.backward()
        optimizer.step()

        # 集計（従来通り：taskはsourceのみで計算）
        correct  += sum((outputs.squeeze(1) > 0.5) == (labels > 0.5)) / len(labels)
        total    += labels.size(0)
        sumlabel += sum((labels > 0.5))
        positive += sum((outputs > 0.5))
        sumout   += sum(outputs)

    train_loss = running_loss
    train_acc  = correct / len(train_loader)
    return None

In [7]:
torch.backends.cudnn.benchmark = True  # 畳み込みの最適化
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

In [8]:
# 利用可能なGPU一覧
n_gpus = torch.cuda.device_count()
print("GPU count:", n_gpus)
for i in range(n_gpus):
    print(i, torch.cuda.get_device_name(i))

# ここでは 6 枚全部を使う前提
DEVICE_IDS = list(range(n_gpus))  # 実際のGPU数に合わせる

# DataParallelの"メイン"になるGPU
MAIN_DEVICE_ID = DEVICE_IDS[0]
device = torch.device(f"cuda:{MAIN_DEVICE_ID}")
torch.cuda.set_device(MAIN_DEVICE_ID)

GPU count: 6
0 NVIDIA RTX A5000
1 NVIDIA RTX A5000
2 NVIDIA RTX A5000
3 NVIDIA RTX A5000
4 NVIDIA RTX A5000
5 NVIDIA RTX A5000


In [9]:
import time
import torch
import torch.nn as nn

def build_model(model):
    # ① モデルインスタンス化（Lazyレイヤ含む）
    # ② まず単一GPUに乗せる
    model = model.to(device)

    # ③ ダミー入力で一度だけ forward して Lazyレイヤを初期化
    with torch.no_grad():
        # ここは実データと同じ形に合わせることが重要
        dummy_size = 2  # 小さくてOK
        dummy_images   = torch.randn(dummy_size, 3, 256, 256, device=device)
        dummy_images2  = torch.randn(dummy_size, 5, 16, 16, device=device)  # 例
        dummy_images3  = torch.randn(dummy_size, 2, 1, 1, device=device)  # 例
        dummy_images3B = torch.randn(dummy_size, 5, 1, 1, device=device)  # 例
        dummy_images4B = torch.randn(dummy_size, 9, 256, 256, device=device)  # 実際のshapeに合わせる
        dummy_images4C = torch.randn(dummy_size, 12, 256, 256, device=device)

        _ = model(dummy_images, dummy_images2, dummy_images3, dummy_images3B,
                  dummy_images4B, dummy_images4C)

    # ④ Lazyパラメータが「普通の Parameter」に変わった状態で DataParallel でラップ
    if torch.cuda.device_count() > 1 and len(DEVICE_IDS) > 1:
        print("Using DataParallel on GPUs:", DEVICE_IDS)
        model = nn.DataParallel(
            model,
            device_ids=DEVICE_IDS,
            output_device=MAIN_DEVICE_ID,
        )

    # ⑤ torch.compile でGPUカーネル最適化（PyTorch 2.0+）
    try:
        model = torch.compile(model, mode="default")
        print("torch.compile applied (mode='reduce-overhead').")
    except Exception as e:
        print(f"torch.compile skipped: {e}")

    return model.to(device)


In [10]:


# 追加: 必要な import
from sklearn.metrics import f1_score, average_precision_score

def _forward_batch(model, batch, device, logits_out=False):
    """
    TensorDataset の最後の要素を y、それ以外を X* として取り出し forward するヘルパ。
    - 複数ブランチ入力に対応: model(*Xs) を試み、失敗すれば model(Xs[0]) にフォールバック。
    - BCEと出力の整合:
        - BCEWithLogitsLoss を使う場合: logits_out=True（Sigmoidしない）
        - BCELoss を使う場合: logits_out=False（Sigmoid済み出力 or 手動でSigmoid）
    """
    *Xs, y = batch
    Xs = [x.float().to(device, dtype=torch.float32, non_blocking=True) for x in Xs]
    y = y.float().to(device, dtype=torch.float32, non_blocking=True)
    
    try:
        out = model(*Xs)
    except TypeError:
        out = model(Xs[0])
    if out.ndim > 1 and out.size(-1) == 1:
        out = out.squeeze(-1)

    if logits_out:               # BCEWithLogitsLoss のとき
        logits = out
        probs = torch.sigmoid(out)
    else:                        # BCELoss のとき（出力がSigmoid済み前提）
        logits = out
        probs = out

    return logits, probs, y

def train_epoch(model, loader, optimizer, criterion, device, logits_out=False):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pos_mean_list, score_mean_list = [], []

    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        logits, probs, y = _forward_batch(model, batch, device, logits_out=logits_out)

        loss = criterion(probs if isinstance(criterion, torch.nn.BCELoss) else logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        preds = (probs >= 0.5).long()
        correct += (preds == y.long()).sum().item()
        total   += y.size(0)
        pos_mean_list.append(y.float().mean().item())
        score_mean_list.append(probs.float().mean().item())

    train_loss = running_loss / max(total, 1)
    train_acc  = correct / max(total, 1)
    train_positive   = float(np.mean(pos_mean_list)) if pos_mean_list else 0.0
    train_meanscore  = float(np.mean(score_mean_list)) if score_mean_list else 0.0

    return train_loss, train_acc, train_positive, train_meanscore

from sklearn.metrics import roc_auc_score

@torch.no_grad()
def eval_epoch(model, loader, criterion, device, logits_out=False, use_streaming_auc=True):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    y_true_all, y_prob_all = [], []
    y_pred_all = []
    pos_mean_list, score_mean_list = [], []

    for batch in loader:
        logits, probs, y = _forward_batch(model, batch, device, logits_out=logits_out)
        loss = criterion(probs if isinstance(criterion, torch.nn.BCELoss) else logits, y)

        running_loss += loss.item() * y.size(0)
        preds = (probs >= 0.5).long()
        correct += (preds == y.long()).sum().item()
        total   += y.size(0)

        # AUC用に確率と正解を蓄積（2値前提：陽性確率は probs そのもの）
        y_true_all.append(y.detach().long().cpu().numpy())
        y_prob_all.append(probs.detach().float().cpu().numpy())
        y_pred_all.append(preds.detach().long().cpu().numpy())
        pos_mean_list.append(y.float().mean().item())
        score_mean_list.append(probs.float().mean().item())

    valid_loss = running_loss / max(total, 1)
    valid_acc  = correct / max(total, 1)

    y_true_all = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_prob_all = np.concatenate(y_prob_all) if y_prob_all else np.array([])
    y_pred_all = np.concatenate(y_pred_all) if y_pred_all else np.array([])

    # --- AUCの算出 ---
    auc_val = float("nan")
    if y_true_all.size > 0:
        try:
            if use_streaming_auc and 'auc_binary_streaming' in globals():
                # 既存のストリーミングAUC関数を優先
                auc_val = auc_binary_streaming(model, loader, device=device, thresholds=1028)
                auc_val = float(getattr(auc_val, "item", lambda: auc_val)())
            else:
                # フォールバック：sklearn
                # 片側しか出現しないfoldでは例外が出るためtry/except
                auc_val = roc_auc_score(y_true_all, y_prob_all)
        except Exception:
            auc_val = float("nan")

    # 参考：F1など他指標を継続して返す場合
    from sklearn.metrics import f1_score, average_precision_score
    valid_f1 = f1_score(y_true_all, y_pred_all) if y_true_all.size > 0 else 0.0

    valid_positive  = float(np.mean(pos_mean_list)) if pos_mean_list else 0.0
    valid_meanscore = float(np.mean(score_mean_list)) if score_mean_list else 0.0

    try:
        ap_val = average_precision_score(y_true_all, y_prob_all) if y_true_all.size > 0 else float("nan")
    except Exception:
        ap_val = float("nan")
    return valid_loss, valid_acc, valid_positive, valid_f1, valid_meanscore, auc_val, ap_val

def evaluation(model, dataset, loss):
    n = len(dataset[1])
    output = 1 * (model(train_dataset[0]) > 0.5)
    percent_of_positive = output / n
    meanloss = loss / n 
    total = sum(1*(output>-9999))
    accuracy = sum(output == dataset[1])/n
    TP = sum(output*dataset[1]) /total; FN = sum((1-output)*dataset[1]) /total; FP = sum(output*(1-dataset[1])) /total
    Precision = TP/(TP+FP); Recall = TP/(TP+FN);F1 = 2*(Precision*Recall) /(Precision+Recall) 
    auc = multiclass_auroc(output,target=labels,average="macro").item()
    return accuracy, meanloss, percent_of_positive, F1,auc

In [11]:
import datetime
from torch.utils.data.dataset import Subset
print(datetime.datetime.now())

2026-06-08 21:56:39.828893


In [ ]:
from torch.utils.data import TensorDataset
Tensor_dataset = torch.load("../../../dataset_tensor_DANN.pt", map_location="cpu")

/tmp/ipykernel_3935201/4001648170.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  Tensor_dataset = torch.load("../../dataset_tensor_DANN.pt", map_location="cpu")


In [ ]:
import os
import time
import math
import gc
import random

import numpy as np
import pandas as pd
import torch
import optuna

from torch.utils.data import DataLoader, Subset
from optuna.pruners import MedianPruner

# =========================================================
# 5. 空間的クロスバリデーションによる学習と評価 + Optuna (DANN)
# =========================================================

CSV_PATH = "results/CNN-DANN-県を予測-optuna.csv"
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

Lcluster = []
LACC = []
Lauc = []

RESULT_COLUMNS = [
    "pref", "n", "epoch",
    "learning_rate", "weight_decay", "batch_size",
    "m1_size", "m2_size", "dropout", "base_channels", "alpha",
    "train_y", "train_acc", "train_positive", "train_meanscore",
    "valid_y", "valid_acc", "valid_auc", "CNN_valid_auc", "valid_positive", "valid_meanscore",
    "test_y", "test_acc", "test_auc", "CNN_test_auc", "test_positive", "test_meanscore"
]

if os.path.exists(CSV_PATH):
    final_df = pd.read_csv(CSV_PATH)
else:
    final_df = pd.DataFrame(columns=RESULT_COLUMNS)
    final_df.to_csv(CSV_PATH, index=False)

num_epochs = 200
patience = 10
WAIT_SECONDS = 1800
MAX_RETRIES_PER_TRIAL = 5


def append_result_row(
    pref,
    best_epoch,
    lr,
    weight_decay,
    batch_size,
    m1_size,
    m2_size,
    dropout,
    base_ch,
    alpha,
    y_train,
    train_acc,
    train_positive,
    train_meanscore,
    y_valid,
    best_val_acc,
    best_val_auc,
    valid_positive,
    valid_meanscore,
    y_test,
    best_test_acc,
    best_test_auc,
    test_positive,
    test_meanscore,
    train_prauc=float("nan"),
    valid_prauc=float("nan"),
    test_prauc=float("nan"),
):
    """結果を CSV に1行追記する。"""
    final_df_local = pd.read_csv(CSV_PATH)

    tmp_df = pd.DataFrame([{
        "pref": pref,
        "n": len(final_df_local),
        "epoch": best_epoch,
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "m1_size": m1_size,
        "m2_size": m2_size,
        "dropout": dropout,
        "base_channels": base_ch,
        "alpha": alpha,
        "train_y": float(np.mean(y_train)),
        "train_acc": float(train_acc),
        "train_positive": float(train_positive),
        "train_meanscore": float(train_meanscore),
        "valid_y": float(np.mean(y_valid)),
        "valid_acc": float(best_val_acc),
        "valid_auc": float(best_val_auc),
        "CNN_valid_auc": float(best_val_auc),
        "valid_positive": float(valid_positive),
        "valid_meanscore": float(valid_meanscore),
        "test_y": float(np.mean(y_test)),
        "test_acc": float(best_test_acc),
        "test_auc": float(best_test_auc),
        "CNN_test_auc": float(best_test_auc),
        "test_positive": float(test_positive),
        "test_meanscore": float(test_meanscore),
        "train_prauc": float(train_prauc),
        "valid_prauc": float(valid_prauc),
        "test_prauc": float(test_prauc),
    }])

    final_df_local = pd.concat([final_df_local, tmp_df], ignore_index=True)
    final_df_local.to_csv(CSV_PATH, index=False)

    return len(final_df_local) - 1

In [ ]:
FOLDS = [("宮城県","山形県"),("山形県","岩手県"),("岩手県","福島県"),("福島県","秋田県"),("秋田県","宮城県")]
for n, (TEST_GRP, VALID_GRP) in enumerate(FOLDS):

    print(f"Training on fold {n}: test={TEST_GRP}, valid={VALID_GRP}")

    ########## データ分割: 実験3整合の持ち回り（test=TEST_GRP / valid=VALID_GRP / train=残り。割合は実験3の県別件数に一致）##########
    SEED = 42
    test_index  = list(dataset.index[dataset["fold_group"] == TEST_GRP])
    valid_index = list(dataset.index[dataset["fold_group"] == VALID_GRP])
    train_index = list(dataset.index[(dataset["fold_group"] != TEST_GRP) & (dataset["fold_group"] != VALID_GRP)])
    random.seed(SEED)  # ダウンサンプリングを再現可能に
    random.shuffle(test_index)
    random.shuffle(valid_index)
    random.shuffle(train_index)

    # train の負例(kuma_view==0)を 1/10 にダウンサンプリング（正例は全て保持）
    _labels = dataset["kuma_view"]
    train_pos = [i for i in train_index if _labels[i] == 1]
    train_neg = [i for i in train_index if _labels[i] == 0]
    random.shuffle(train_neg)
    train_neg = train_neg[: len(train_neg) // 10]
    train_index = train_pos + train_neg
    random.shuffle(train_index)

    train_idx = torch.as_tensor(train_index, dtype=torch.long)
    valid_idx = torch.as_tensor(valid_index, dtype=torch.long)
    test_idx = torch.as_tensor(test_index, dtype=torch.long)

    train_dataset = Subset(Tensor_dataset, train_idx.tolist())
    valid_dataset = Subset(Tensor_dataset, valid_idx.tolist())
    test_dataset = Subset(Tensor_dataset, test_idx.tolist())

    y_train = dataset["kuma_view"][train_index]
    y_valid = dataset["kuma_view"][valid_index]
    y_test = dataset["kuma_view"][test_index]

    _GLOBAL_BEST_AUC = [-1.0]
    def objective(trial: optuna.Trial):
        for attempt in range(1, MAX_RETRIES_PER_TRIAL + 1):
            model = None
            optimizer = None

            train_loader_for_train = None
            target_loader_for_train = None
            train_loader = None
            valid_loader = None
            test_loader = None

            try:
                print(f"[Trial {trial.number}] Attempt {attempt}/{MAX_RETRIES_PER_TRIAL}")
                print(
                    "y_train:", float(np.mean(y_train)),
                    "y_valid:", float(np.mean(y_valid)),
                    "y_test:", float(np.mean(y_test))
                )

                # 記録用
                train_loss_list = []
                train_acc_list = []
                train_positive_list = []
                train_F1_list = []
                train_meanscore_list = []
                train_auc_list = []
                train_prauc_list = []; valid_prauc_list = []; test_prauc_list = []

                valid_loss_list = []
                valid_acc_list = []
                valid_positive_list = []
                valid_F1_list = []
                valid_auc_list = []
                valid_meanscore_list = []

                test_loss_list = []
                test_acc_list = []
                test_positive_list = []
                test_F1_list = []
                test_auc_list = []
                test_meanscore_list = []

                # 探索パラメータ
                lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
                weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
                batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
                dropout = trial.suggest_float("dropout", 0.0, 0.3)
                base_ch = trial.suggest_categorical("base_channels", [16, 32])
                m1_size = trial.suggest_categorical("m1_size", [512, 1024, 2048, 4096])
                m2_size = trial.suggest_categorical("m2_size", [256, 512, 1024])
                alpha = trial.suggest_float("alpha", 0.1, 1.0)

                domain_criterion = torch.nn.CrossEntropyLoss()

                # DataLoader
                train_loader_for_train = DataLoader(
                    train_dataset,
                    batch_size=batch_size * 6,
                    shuffle=True,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=True,
                )
                target_loader_for_train = DataLoader(
                    test_dataset,
                    batch_size=batch_size * 6,
                    shuffle=True,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=True,
                )
                train_loader = DataLoader(
                    train_dataset,
                    batch_size=256,
                    shuffle=False,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=True,
                )
                valid_loader = DataLoader(
                    valid_dataset,
                    batch_size=256,
                    shuffle=False,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=True,
                )
                test_loader = DataLoader(
                    test_dataset,
                    batch_size=256,
                    shuffle=False,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=True,
                )

                # モデル
                model = DANN_CNN(
                    domain_classes=5,
                    base_channels=base_ch,
                    dropout=dropout,
                    m1=m1_size,
                    m2=m2_size,
                ).float()

                model = build_model(model)

                criterion = torch.nn.BCEWithLogitsLoss()
                optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

                best_val_auc = -1.0
                best_val_acc = 0.0
                best_test_auc = 0.0
                best_test_acc = 0.0
                best_epoch = -1
                best_val_prauc = float("nan"); best_test_prauc = float("nan"); best_train_prauc = float("nan")
                wait = 0

                last_train_acc = 0.0
                last_train_positive = 0.0
                last_train_meanscore = 0.0
                last_valid_positive = 0.0
                last_valid_meanscore = 0.0
                last_test_positive = 0.0
                last_test_meanscore = 0.0

                for epoch in range(num_epochs):
                    p = epoch / max(num_epochs - 1, 1)
                    grl_lambda = 2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0

                    model.train()
                    train(
                        model,
                        train_loader_for_train,
                        criterion,
                        device,
                        optimizer,
                        target_loader=target_loader_for_train,
                        domain_criterion=domain_criterion,
                        alpha=alpha,
                        grl_lambda=grl_lambda,
                        use_amp=True,
                    )

                    model.eval()
                    train_loss, train_acc, train_positive, train_F1, train_meanscore, train_auc, train_prauc = \
                        eval_epoch(model, train_loader, criterion, device, logits_out=False, use_streaming_auc=True)
                    valid_loss, valid_acc, valid_positive, valid_F1, valid_meanscore, valid_auc, valid_prauc = \
                        eval_epoch(model, valid_loader, criterion, device, logits_out=False, use_streaming_auc=True)
                    test_loss, test_acc, test_positive, test_F1, test_meanscore, test_auc, test_prauc = \
                        eval_epoch(model, test_loader, criterion, device, logits_out=False, use_streaming_auc=True)

                    # すべて float 化しておくと、CSV 保存時や print 時に安全
                    train_loss = float(train_loss)
                    train_acc = float(train_acc)
                    train_positive = float(train_positive)
                    train_F1 = float(train_F1)
                    train_meanscore = float(train_meanscore)
                    train_auc = float(train_auc)

                    valid_loss = float(valid_loss)
                    valid_acc = float(valid_acc)
                    valid_positive = float(valid_positive)
                    valid_F1 = float(valid_F1)
                    valid_meanscore = float(valid_meanscore)
                    valid_auc = float(valid_auc)

                    test_loss = float(test_loss)
                    test_acc = float(test_acc)
                    test_positive = float(test_positive)
                    test_F1 = float(test_F1)
                    test_meanscore = float(test_meanscore)
                    test_auc = float(test_auc)

                    train_loss_list.append(train_loss)
                    train_acc_list.append(train_acc)
                    train_positive_list.append(train_positive)
                    train_F1_list.append(train_F1)
                    train_meanscore_list.append(train_meanscore)
                    train_auc_list.append(train_auc)

                    valid_loss_list.append(valid_loss)
                    valid_acc_list.append(valid_acc)
                    valid_positive_list.append(valid_positive)
                    valid_F1_list.append(valid_F1)
                    valid_meanscore_list.append(valid_meanscore)
                    valid_auc_list.append(valid_auc)

                    test_loss_list.append(test_loss)
                    test_acc_list.append(test_acc)
                    test_positive_list.append(test_positive)
                    test_F1_list.append(test_F1)
                    test_meanscore_list.append(test_meanscore)
                    test_auc_list.append(test_auc); train_prauc_list.append(train_prauc); valid_prauc_list.append(valid_prauc); test_prauc_list.append(test_prauc)

                    last_train_acc = train_acc
                    last_train_positive = train_positive
                    last_train_meanscore = train_meanscore
                    last_valid_positive = valid_positive
                    last_valid_meanscore = valid_meanscore
                    last_test_positive = test_positive
                    last_test_meanscore = test_meanscore

                    if valid_auc > best_val_auc:
                        best_val_auc = valid_auc
                        best_val_acc = valid_acc
                        best_test_auc = test_auc
                        best_test_acc = test_acc
                        best_epoch = epoch + 1
                        best_val_prauc = valid_prauc; best_test_prauc = test_prauc; best_train_prauc = train_prauc
                        # --- per-trial best-epoch モデル保存（state_dict＋メタ情報）---
                        _m = model.module if isinstance(model, torch.nn.DataParallel) else model
                        _pkg = {"model": "cnn_dann", "trial": int(trial.number), "epoch": int(best_epoch),
                                "valid_auc": float(best_val_auc), "params": dict(trial.params),
                                "state_dict": {k: v.detach().cpu().clone() for k, v in _m.state_dict().items()}}
                        os.makedirs("optuna_models", exist_ok=True)
                        torch.save(_pkg, f"optuna_models/cnn_dann_trial{int(trial.number)}_best.pkl")
                        if float(best_val_auc) > _GLOBAL_BEST_AUC[0]:
                            _GLOBAL_BEST_AUC[0] = float(best_val_auc)
                            torch.save(_pkg, "optuna_models/cnn_dann_best.pkl")
                        wait = 0
                    else:
                        wait += 1

                    # Optuna の pruning 判定には report が必要
                    trial.report(valid_auc, step=epoch)

                    print(
                        f"n = {n} | epoch {epoch+1:03d} | "
                        f"valid_auc {valid_auc:.4f} | test_auc {test_auc:.4f} | "
                        f"best {best_val_auc:.4f} | wait {wait}"
                    )

                    # Optuna pruning
                    if trial.should_prune():
                        row_id = append_result_row(
                            pref=n,
                            best_epoch=best_epoch,
                            lr=lr,
                            weight_decay=weight_decay,
                            batch_size=batch_size,
                            m1_size=m1_size,
                            m2_size=m2_size,
                            dropout=dropout,
                            base_ch=base_ch,
                            alpha=alpha,
                            y_train=y_train,
                            train_acc=last_train_acc,
                            train_positive=last_train_positive,
                            train_meanscore=last_train_meanscore,
                            y_valid=y_valid,
                            best_val_acc=best_val_acc,
                            best_val_auc=best_val_auc,
                            valid_positive=last_valid_positive,
                            valid_meanscore=last_valid_meanscore,
                            y_test=y_test,
                            best_test_acc=best_test_acc,
                            best_test_auc=best_test_auc,
                            test_positive=last_test_positive,
                            test_meanscore=last_test_meanscore,
                            train_prauc=best_train_prauc,
                            valid_prauc=best_val_prauc,
                            test_prauc=best_test_prauc,
                        )

                        datalist = [
                            train_loss_list, valid_loss_list, test_loss_list,
                            train_acc_list, valid_acc_list, test_acc_list,
                            train_positive_list, valid_positive_list, test_positive_list,
                            train_F1_list, valid_F1_list, test_F1_list,
                            train_auc_list, valid_auc_list, test_auc_list,
                            train_meanscore_list, valid_meanscore_list, test_meanscore_list, train_prauc_list, valid_prauc_list, test_prauc_list,
                        ]

                        # ここが save_metrics の誤記なら修正してください
                        seve_metrics(f"CNN-DANN-県を予測-optuna-{row_id}", datalist=datalist)
                        raise optuna.TrialPruned()

                    # 早期終了は prune ではなく正常終了扱いにする
                    if wait >= patience or valid_auc < 0.501:
                        row_id = append_result_row(
                            pref=n,
                            best_epoch=best_epoch,
                            lr=lr,
                            weight_decay=weight_decay,
                            batch_size=batch_size,
                            m1_size=m1_size,
                            m2_size=m2_size,
                            dropout=dropout,
                            base_ch=base_ch,
                            alpha=alpha,
                            y_train=y_train,
                            train_acc=last_train_acc,
                            train_positive=last_train_positive,
                            train_meanscore=last_train_meanscore,
                            y_valid=y_valid,
                            best_val_acc=best_val_acc,
                            best_val_auc=best_val_auc,
                            valid_positive=last_valid_positive,
                            valid_meanscore=last_valid_meanscore,
                            y_test=y_test,
                            best_test_acc=best_test_acc,
                            best_test_auc=best_test_auc,
                            test_positive=last_test_positive,
                            test_meanscore=last_test_meanscore,
                        )

                        datalist = [
                            train_loss_list, valid_loss_list, test_loss_list,
                            train_acc_list, valid_acc_list, test_acc_list,
                            train_positive_list, valid_positive_list, test_positive_list,
                            train_F1_list, valid_F1_list, test_F1_list,
                            train_auc_list, valid_auc_list, test_auc_list,
                            train_meanscore_list, valid_meanscore_list, test_meanscore_list, train_prauc_list, valid_prauc_list, test_prauc_list,
                        ]

                        seve_metrics(f"CNN-DANN-県を予測-optuna-{row_id}", datalist=datalist)

                        del model
                        del optimizer
                        torch.cuda.empty_cache()
                        gc.collect()

                        return best_val_auc

                # 正常終了
                row_id = append_result_row(
                    pref=n,
                    best_epoch=best_epoch,
                    lr=lr,
                    weight_decay=weight_decay,
                    batch_size=batch_size,
                    m1_size=m1_size,
                    m2_size=m2_size,
                    dropout=dropout,
                    base_ch=base_ch,
                    alpha=alpha,
                    y_train=y_train,
                    train_acc=last_train_acc,
                    train_positive=last_train_positive,
                    train_meanscore=last_train_meanscore,
                    y_valid=y_valid,
                    best_val_acc=best_val_acc,
                    best_val_auc=best_val_auc,
                    valid_positive=last_valid_positive,
                    valid_meanscore=last_valid_meanscore,
                    y_test=y_test,
                    best_test_acc=best_test_acc,
                    best_test_auc=best_test_auc,
                    test_positive=last_test_positive,
                    test_meanscore=last_test_meanscore,
                )

                datalist = [
                    train_loss_list, valid_loss_list, test_loss_list,
                    train_acc_list, valid_acc_list, test_acc_list,
                    train_positive_list, valid_positive_list, test_positive_list,
                    train_F1_list, valid_F1_list, test_F1_list,
                    train_auc_list, valid_auc_list, test_auc_list,
                    train_meanscore_list, valid_meanscore_list, test_meanscore_list, train_prauc_list, valid_prauc_list, test_prauc_list,
                ]

                seve_metrics(f"CNN-DANN-県を予測-optuna-{row_id}", datalist=datalist)

                del model
                del optimizer
                torch.cuda.empty_cache()
                gc.collect()

                return best_val_auc

            except torch.cuda.OutOfMemoryError as e:
                print(f"[Trial {trial.number}] CUDA OOM on attempt {attempt}/{MAX_RETRIES_PER_TRIAL}: {e}")

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"[Trial {trial.number}] RuntimeError OOM on attempt {attempt}/{MAX_RETRIES_PER_TRIAL}: {e}")
                else:
                    raise

            # OOM 後の後始末
            model = None
            optimizer = None
            train_loader_for_train = None
            target_loader_for_train = None
            train_loader = None
            valid_loader = None
            test_loader = None

            torch.cuda.empty_cache()
            gc.collect()

            if attempt == MAX_RETRIES_PER_TRIAL:
                print(f"[Trial {trial.number}] Reached max OOM retries. Pruning trial.")
                raise optuna.TrialPruned()

            print(f"[Trial {trial.number}] Waiting {WAIT_SECONDS} seconds before retry...")
            time.sleep(WAIT_SECONDS)

    study = optuna.create_study(
        study_name=f"DANN_fold_{n}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
    )
    study.optimize(objective, n_trials=5, gc_after_trial=True, n_jobs=1)